In [ ]:
import os
import sys
import argparse
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
from scipy import sparse
from scipy.stats import rankdata as _rankdata
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

warnings.filterwarnings("ignore")

In [ ]:
ref_adata = sc.read_h5ad('oral_healthy_reference/health_Buccal-Gingival.h5ad')
ref_adata.obs['celltype'] = ref_adata.obs['paperLabels']
sp_adata = sc.read_h5ad('analysis_base.h5ad')
sp_adata = sp_adata[sp_adata.obs['slide'] == '4']
sp_adata

In [ ]:
common = ref_adata.var_names.intersection(sp_adata.var_names)
print(f"Ref: {ref_adata.n_vars}, Query: {sp_adata.n_vars}, Shared: {len(common)} ({100*len(common)/min(ref_adata.n_vars, sp_adata.n_vars):.0f}%)")
ref_adata = ref_adata[:, common].copy()
sp_adata = sp_adata[:, common].copy()

In [ ]:
def counts_to_rank(adata, desc="Ranking genes"):
    X = adata.layers["counts"] if "counts" in adata.layers else adata.X
    if sparse.issparse(X):
        X = X.toarray()
    X = X.astype(np.float32)
    ranks = np.zeros_like(X, dtype=np.float32)
    for i in tqdm(range(X.shape[0]), desc=desc, leave=False):
        row = X[i]
        nz = row > 0
        if nz.any():
            ranks[i, nz] = _rankdata(-row[nz], method="dense").astype(np.float32)
    return ranks

celltype_col = 'celltype'
X_train = counts_to_rank(ref_adata, "Ranking ref")
y_train = ref_adata.obs[celltype_col].astype(str).values
X_query = counts_to_rank(sp_adata, "Ranking query")

In [ ]:
%%time

le = LabelEncoder()
y_enc = le.fit_transform(y_train)
sample_weights = compute_sample_weight("balanced", y_enc)

xgb_params = dict(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="mlogloss", n_jobs=-1, device="cuda", random_state=42,
)

# CV with per-fold progress
skf = StratifiedKFold(3, shuffle=True, random_state=42)
cv_scores = []
for i, (train_idx, val_idx) in enumerate(skf.split(X_train, y_enc)):
    print(f"--- Fold {i+1}/3 ---")
    fold_clf = xgb.XGBClassifier(**xgb_params)
    fold_clf.fit(X_train[train_idx], y_enc[train_idx], verbose=50)
    score = balanced_accuracy_score(y_enc[val_idx], fold_clf.predict(X_train[val_idx]))
    print(f"Fold {i+1} balanced accuracy: {score:.3f}\n")
    cv_scores.append(score)

print(f"CV balanced accuracy: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}")

# Final fit on all data
print("--- Final fit ---")
clf = xgb.XGBClassifier(**xgb_params)
clf.fit(X_train, y_enc, sample_weight=sample_weights, verbose=50)
print("Done.")

In [ ]:
proba = clf.predict_proba(X_query)
sp_adata.obs["predicted_cell_type"] = le.inverse_transform(np.argmax(proba, axis=1))
sp_adata.obs["predicted_cell_type_confidence"] = proba.max(axis=1)

In [ ]:
# distribution
sp_adata.obs["predicted_cell_type"].value_counts().plot.barh(figsize=(8,6), title="Predicted cell types")
# low-confidence cells — worth inspecting
low_conf = sp_adata.obs["predicted_cell_type_confidence"] < 0.5
print(f"Low confidence (<0.5): {low_conf.sum()} / {sp_adata.n_obs} ({100*low_conf.mean():.1f}%)")

In [ ]:
# Check distribution of confidence.
sp_adata.obs["predicted_cell_type_confidence"].hist(bins=50, figsize=(8,4))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = sp_adata.obs.groupby("predicted_cell_type")["predicted_cell_type_confidence"].median().sort_values(ascending=False).index
palette = sns.color_palette("husl", n_colors=len(order))

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(
    data=sp_adata.obs, x="predicted_cell_type", y="predicted_cell_type_confidence",
    order=order, palette=palette, fliersize=0, ax=ax
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_title("Prediction confidence by cell type")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
import squidpy as sq
sq.pl.spatial_scatter(sp_adata[sp_adata.obs['core_id'] == 71], shape = None, color = "predicted_cell_type_confidence")